# 강의 04 · 실습 6 — 커스텀 MCP 서버 · (4) 고난도 I

## 1. 문제상황

- 쇼핑몰 재고 시스템은 수량 필드에 사람이 메모를 섞어 적습니다.
- "37 (예약 5)"는 총재고 37에 예약 5가 걸려 있다는 뜻이고, "품절"은 재고가 없다는 뜻이며, "1,200"처럼 쉼표가 붙기도 합니다.
- 이 값을 그대로 정수 반환 도구로 내보내면 서버 쪽 검증에서 호출이 깨지고, 모델은 오류 문자열 안의 숫자를 읽어 "구매 가능 32개"처럼 도구가 준 적 없는 숫자를 답합니다.
- 게다가 등록되지 않은 품목을 물으면 조회가 예외를 내는데, 그 오류도 최종 답에서는 보이지 않습니다.
- 상담 화면에는 오류가 없고 고객은 그 숫자를 믿습니다.

## 2. 문제와 목표

- **문제**: 재고 시스템의 원래 값이 정수 계약과 어긋나 도구 호출이 깨지고, 깨진 호출의 오류 문자열이 그대로 모델에 들어가 모델이 숫자를 지어냅니다. 없는 품목의 조회 오류도 최종 답에 묻힙니다.
- **목표**: 재고 원래 값을 도구 안에서 정규화해 계약대로 돌려주는 서버 `stock`을 만들고, 클라이언트 쪽에서는 도구 결과 메시지를 검사해 오류가 있으면 최종 답과 함께 경고를 출력하게 합니다.
    - 서버의 도구 두 개: `get_stock(item)`는 원래 값에서 괄호 앞부분만 남기고 쉼표를 지운 뒤 숫자이면 정수, 아니면 `None`을 돌려주고, `list_items()`는 등록된 품목 이름 목록을 돌려줍니다. 등록되지 않은 품목은 `KeyError`가 나도록 둡니다.
    - 재고 원래 값: 품목 이름을 문자열(`"37 (예약 5)"`·`"품절"`·`"1,200"`)에 대응시키는 딕셔너리 `STOCK_DB`이며, 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 클라이언트의 검사: 실행 결과의 도구 결과 메시지 중 오류인 것을 세어 경고를 출력하는 함수 `check_errors`입니다. 질문 세 문장(무선 마우스·키보드·태블릿)은 코드에 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 「무선 마우스」 질문에 `get_stock`이 오류 없이 37을 돌려줍니다.
    - 「키보드」 질문에 `None`을 돌려주어(도구 결과 내용이 비어 있음) 모델이 재고 숫자를 말하지 않습니다.
    - 「태블릿」(없는 품목) 질문에 도구 호출이 오류로 끝나고 클라이언트가 그 오류를 세어 경고를 출력하는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex06_s4_diagram.svg)

## 4. 단계별 요구사항

1. **서버 인스턴스를 만듭니다.**
    - `FastMCP`로 이름이 `stock`인 서버 객체를 만듭니다.
    - 서버 파일 이름은 `stock_server.py`입니다.
2. **함수를 도구로 등록합니다.**
    - `STOCK_DB`는 품목 이름을 재고 시스템의 원래 값 문자열에 대응시킵니다.
    - 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - `get_stock(item: str) -> int | None`은 원래 값의 괄호 앞부분만 남기고 쉼표를 지운 뒤, 숫자이면 정수로 돌려주고 숫자가 아니면 `None`을 돌려줍니다.
    - 등록되지 않은 품목은 `KeyError`가 나도록 그대로 둡니다(도구 실행 오류를 관찰하기 위해서입니다). `list_items() -> list[str]`은 등록된 품목 이름 목록을 돌려줍니다.
    - 두 함수 모두 `@mcp.tool()`로 등록하고 독스트링을 답니다.
3. **서버를 시작합니다.**
    - `__main__` 가드 안에서 `mcp.run(transport="stdio")`를 호출합니다.
4. **클라이언트에 검사 단계를 더합니다.**
    - `check_errors(result)` 함수는 실행 결과의 메시지 중 `ToolMessage`이면서 `status`가 `'error'`인 것을 세어, 있으면 「[경고] 도구 오류 N개 — 최종 답의 숫자를 믿지 않습니다」를 출력하고 개수를 돌려줍니다.
5. **클라이언트에서 띄워 확인합니다.**
    - 도구 목록을 출력한 뒤 「무선 마우스 재고 있나요?」·「키보드 재고 있나요?」·「태블릿 재고 있나요?」를 차례로 물어, 질문마다 메시지 기록을 출력하고 `check_errors`를 호출합니다.
    - 도구 목록은 「서버가 준 도구:」 줄로 출력합니다.

## 5. 코드 골격 — MCP 서버 3단

FastMCP로 서버를 세우는 순서는 다음 세 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 세 단계와 하나씩 대응합니다. 서버 코드는 `%%writefile`로 파일에 쓰고, 단계 ③의 확인 셀에서 클라이언트가 그 파일을 실행 명령으로 띄웁니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 인스턴스 생성 | 서버 객체를 이름과 함께 만듭니다 | `mcp = FastMCP("stock")` | 1 |
| ② 도구 등록 | 파이썬 함수에 표시를 붙이고, 타입 힌트와 설명 한 줄을 답니다 | `@mcp.tool()`, `def get_stock(item: str) -> int | None` | 2 |
| ③ 서버 시작 | 표준입출력으로 말하도록 지정해 서버를 띄웁니다. 클라이언트가 실행 명령으로 띄우고 도구 목록을 받아 씁니다 | `mcp.run(transport="stdio")`, `MultiServerMCPClient`, `get_tools()` | 3, 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

클라이언트 쪽 준비입니다. 라이브러리를 불러오고 모델을 준비하고, 도구 호출 루프 `build_loop`와 연결 선언 함수 `server_config`를 정의합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `build_loop`는 받아 온 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결하는 도구 호출 루프입니다. 서버 코드가 아니라 서버를 쓰는 쪽의 코드입니다.
- `sys.stderr = sys.__stderr__` 줄은 노트북 전용입니다. 노트북 커널은 표준 오류 스트림을 화면용 객체로 바꿔 두는데, 서버 프로세스를 띄우는 코드는 원래의 표준 오류 스트림을 요구하므로 되돌려 놓습니다. 이 줄은 MCP 클라이언트를 불러오기 전에 있어야 합니다.
- `server_config`는 서버 파일 하나를 표준입출력으로 띄우는 연결 선언입니다. `sys.executable`은 지금 돌고 있는 파이썬 러너입니다. `FASTMCP_LOG_LEVEL`은 서버의 안내 로그가 화면을 채우지 않게 하는 설정입니다.
- 실행 결과 출력은 `show(result)`, 메시지 글자 추출은 `text_of(m)`로 합니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 커널의 stderr에는 fileno()가 없어 서버 프로세스 시작이 실패하므로 원래 stderr로 되돌린다
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_loop(tools):
    """도구 호출 루프. 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결한다."""
    bound = llm.bind_tools(tools)

    def call_model(state: State) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: State) -> str:
        return "tools" if state["messages"][-1].tool_calls else END

    g = StateGraph(State)
    g.add_node("model", call_model)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
    g.add_edge("tools", "model")
    return g.compile()


def text_of(m) -> str:
    """메시지 내용이 콘텐츠 블록 목록이면 글자 부분만 이어 붙인다."""
    if isinstance(m.content, list):
        return " ".join(p.get("text", "") for p in m.content if isinstance(p, dict))
    return str(m.content)


def show(result) -> None:
    """실행 결과의 메시지를 종류·도구 호출·상태와 함께 한 줄씩 출력한다."""
    for m in result["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif kind == "ToolMessage":
            print(f"[{kind}] status={m.status!r} {text_of(m)[:160]}")
        elif m.content:
            print(f"[{kind}] {text_of(m)[:300]}")


def server_config(file: str) -> dict:
    """서버 파일 하나를 표준입출력으로 띄우는 연결 선언을 만든다."""
    return {"command": sys.executable, "args": [str(Path(file).resolve())],
            "transport": "stdio", "env": {"FASTMCP_LOG_LEVEL": "ERROR"}}


print("클라이언트 준비를 마쳤습니다.")

# 주어진 자료 — 서버 파일(stock_server.py)의 재고 원래 값. 단계 ② 셀에 그대로 옮겨 적는다
STOCK_DB = {"무선 마우스": "37 (예약 5)", "키보드": "품절", "모니터": "1,200"}   # 재고 시스템의 원래 값


### 단계 ① — 인스턴스 생성 (요구사항 1)

`FastMCP(이름)`이 서버 한 대입니다. 괄호 안 이름이 이 서버의 이름입니다. `%%writefile`이 이 셀의 내용을 서버 파일로 저장합니다. 서버 파일은 사람이 직접 실행하지 않고, 단계 ③에서 클라이언트가 띄웁니다.

서버 파일 첫 줄은 `from mcp.server.fastmcp import FastMCP`입니다.


In [ ]:
%%writefile stock_server.py
# 여기에 단계 ①(서버 인스턴스 생성)을 작성합니다.

### 단계 ② — 도구 등록 (요구사항 2)

`@mcp.tool()` 한 줄을 얹으면 그 파이썬 함수가 MCP 도구가 됩니다. 함수 이름이 도구 이름, 독스트링이 도구 설명, 인자와 반환의 타입 힌트가 도구 규격으로 그대로 나갑니다. `%%writefile -a`는 서버 파일 뒤에 이어 붙입니다.

In [ ]:
%%writefile -a stock_server.py
# 여기에 단계 ②(도구 등록)를 작성합니다.

### 단계 ③ — 서버 시작 (요구사항 3, 4, 5)

`transport="stdio"`는 표준입출력으로 말한다는 뜻입니다. 서버는 포트를 열지 않고, 자신을 실행한 쪽과 입출력으로 주고받습니다. 아래 두 셀(③-a, ③-b)이 모두 이 단계에 속합니다.

#### 단계 ③-a — 시작 코드를 서버 파일에 붙입니다

In [ ]:
%%writefile -a stock_server.py
# 여기에 단계 ③-a(서버 시작)를 작성합니다.

#### 단계 ③-b — 클라이언트에서 띄우고 확인합니다

클라이언트에 적는 것은 서버를 띄우는 실행 명령뿐입니다. 서버 코드는 클라이언트에 등장하지 않습니다. `get_tools()`가 서버가 내놓은 도구 목록을 받아 오고, 그 목록을 `build_loop`에 그대로 넣습니다.

클라이언트는 `MultiServerMCPClient({"stock": server_config("stock_server.py")})`로 열고, 도구 목록은 `await client.get_tools()`, 실행은 `await app.ainvoke(...)`로 부릅니다.


In [ ]:
# 여기에 단계 ③-b(클라이언트에서 띄우기, 도구 목록 확인, 오류 검사 함수 check_errors, 루프 실행)를 작성합니다.

## 7. 실행 결과 확인

위 실행 결과에서 다음을 확인합니다.

1. `서버가 준 도구:` 줄에 `get_stock`과 `list_items`가 설명과 함께 있습니다.
2. 「무선 마우스」 질문에서 `ToolMessage`가 `status='success'`이고 값이 37입니다. 원래 값 `"37 (예약 5)"`가 서버 안에서 37로 정규화되었습니다. 최종 답에 32 같은 도구가 주지 않은 숫자가 없습니다.
3. 「키보드」 질문에서 `ToolMessage`가 `status='success'`이지만 내용이 비어 있습니다(`None`). 모델은 재고 숫자를 말하지 않고 확인할 수 없다고 답합니다. 원래 값 `"품절"`이 지어낸 숫자로 바뀌지 않았습니다.
4. 「태블릿」 질문에서 `ToolMessage`가 `status='error'`이고 `KeyError`가 보이며, 그 아래 「[경고] 도구 오류 1개」 줄이 출력됩니다. 최종 답이 무엇이든 경고가 함께 나옵니다.